# LeetCode 245 — Shortest Word Distance III bằng Python

Link đề: https://leetcode.com/problems/shortest-word-distance-iii/ · Độ khó: **Medium** · Chủ đề: Array, String

Skeleton LeetCode:

```python
class Solution:
    def shortestWordDistance(self, wordsDict: List[str], word1: str, word2: str) -> int:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập duyệt mọi cặp chỉ số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng chuỗi `wordsDict` và hai chuỗi `word1`, `word2` có trong mảng; tìm **khoảng cách ngắn nhất** (`|i − j|`, hai chỉ số **khác nhau**) giữa một lần xuất hiện của `word1` và một lần xuất hiện của `word2`. **Điểm mới so với phần I/II:** `word1` có thể **bằng** `word2` (khi đó cần hai lần xuất hiện **khác nhau** của cùng một từ).
2. Biết **ghi nhớ chỉ số xuất hiện gần nhất** khi duyệt một lần, và vì sao cách đơn giản cho `word1 ≠ word2` **hỏng** khi `word1 == word2` (cùng chỉ số bị tính hai lần, ra khoảng cách `0`).
3. Đi từ **duyệt mọi cặp chỉ số `O(n²)`** → **danh sách chỉ số rồi tách hai trường hợp** → **một lượt duyệt tách hai trường hợp (cách tham khảo)** → **một lượt duyệt thống nhất bằng "kiểm tra trước, cập nhật sau"** (không cần rẽ nhánh theo `word1 == word2`).
4. Nhận ra quy tắc vàng khi hai vai có thể trùng: **cập nhật khoảng cách bằng chỉ số xuất hiện *trước đó* rồi mới ghi nhận chỉ số hiện tại**.
5. Tránh các lỗi kinh điển: cập nhật đồng thời hai chỉ số cùng một vị trí (ra `0`), dùng `elif` khiến chỉ số thứ hai không bao giờ được cập nhật, khởi tạo chỉ số bằng `0` thay vì `-1`, và chỉ xét hai lần xuất hiện đầu tiên.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `wordsDict` và hai chuỗi `word1`, `word2` **đã có trong mảng**, hãy trả về **khoảng cách ngắn nhất giữa sự xuất hiện của hai từ này** trong danh sách.

Lưu ý: `word1` và `word2` **có thể giống nhau**. Đề đảm bảo chúng đại diện cho **hai từ riêng biệt** trong danh sách (nghĩa là nếu giống nhau thì từ đó xuất hiện ít nhất hai lần ở hai chỉ số khác nhau).

**Ví dụ 1**

- Input: `wordsDict = ["practice", "makes", "perfect", "coding", "makes"]`, `word1 = "makes"`, `word2 = "coding"`
- Output: `1`

**Ví dụ 2**

- Input: `wordsDict = ["practice", "makes", "perfect", "coding", "makes"]`, `word1 = "makes"`, `word2 = "makes"`
- Output: `3`
- Giải thích: `"makes"` xuất hiện ở chỉ số `1` và `4`; khoảng cách `|4 − 1| = 3`.

**Ràng buộc:** `1 <= wordsDict.length <= 10⁵`; `1 <= wordsDict[i].length <= 10`; `wordsDict[i]` gồm chữ cái thường; `word1` và `word2` có trong `wordsDict`.

In [1]:
from typing import List
import random
from math import inf

WORDS = ["practice", "makes", "perfect", "coding", "makes"]
examples = [
    (WORDS, "makes", "coding", 1),
    (WORDS, "makes", "makes", 3),
]
for words, w1, w2, expected in examples:
    print(f"word1={w1!r}, word2={w2!r}, kỳ vọng {expected}")

word1='makes', word2='coding', kỳ vọng 1
word1='makes', word2='makes', kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Chỉ số, `enumerate` và so sánh chuỗi

`for k, w in enumerate(words)` duyệt cả chỉ số `k` và từ `w`. So sánh hai chuỗi bằng `==`. Ở bài này ta quan tâm **chỉ số** nơi từ xuất hiện chứ không phải bản thân từ.

In [2]:
for k, w in enumerate(WORDS):
    print(k, w, "<- makes" if w == "makes" else "")

0 practice 
1 makes <- makes
2 perfect 
3 coding 
4 makes <- makes


### 3.2. Khoảng cách giữa hai chỉ số: `abs`

Khoảng cách giữa hai lần xuất hiện ở chỉ số `i` và `j` là `|i − j|`. Ta tìm giá trị **nhỏ nhất** trên mọi cặp lần xuất hiện hợp lệ. Hợp lệ nghĩa là `i ≠ j` (hai lần xuất hiện **khác nhau**) — điều này quan trọng khi `word1 == word2`.

In [3]:
a = [i for i, w in enumerate(WORDS) if w == "makes"]       # [1, 4]
b = [i for i, w in enumerate(WORDS) if w == "coding"]      # [3]
print([abs(i - j) for i in a for j in b])                   # [2, 1] -> nhỏ nhất 1
print([abs(i - j) for i in a for j in a if i != j])         # [3, 3]: cùng một từ, i != j -> 3
print([abs(i - j) for i in a for j in a])                   # [0, 3, 3, 0]: nếu quên i != j sẽ ra 0

[2, 1]
[3, 3]
[0, 3, 3, 0]


### 3.3. Hai từ khác nhau: ghi nhớ chỉ số gần nhất của mỗi từ

Duyệt từ trái sang phải; giữ `last1` (chỉ số gần nhất đã gặp của `word1`) và `last2` (của `word2`). Mỗi khi gặp một trong hai từ ta so với chỉ số gần nhất của từ kia. Lý do chỉ cần chỉ số **gần nhất**: với cặp có `word2` ở chỉ số `j`, thì `word1` gần `j` nhất ở phía trước chính là lần xuất hiện **gần nhất trước `j`**.

In [4]:
def shortest_two_words(words, word1, word2):
    last1 = last2 = -1
    best = len(words)
    for k, w in enumerate(words):
        if w == word1:
            last1 = k
        elif w == word2:
            last2 = k
        if last1 != -1 and last2 != -1:
            best = min(best, abs(last1 - last2))
    return best


print(shortest_two_words(WORDS, "makes", "coding"))             # 1
print(shortest_two_words(WORDS, "practice", "perfect"))        # 2

1
2


### 3.4. Hai từ **giống nhau**: chỉ cần chỉ số trước đó của cùng từ

Khi `word1 == word2`, cặp cần tìm là hai lần xuất hiện **khác nhau** của cùng một từ; khoảng cách nhỏ nhất luôn nằm giữa **hai lần xuất hiện liên tiếp**. Vậy chỉ cần giữ `last` (lần xuất hiện trước) và tính `k − last` mỗi khi gặp lại từ đó.

In [5]:
def shortest_same_word(words, word):
    last = -1
    best = len(words)
    for k, w in enumerate(words):
        if w == word:
            if last != -1:
                best = min(best, k - last)
            last = k
    return best


print(shortest_same_word(WORDS, "makes"))              # 3
print(shortest_same_word(["a", "x", "a", "a"], "a"))    # 1 (chỉ số 2 và 3)

3
1


### 3.5. Vì sao cách của mục 3.3 hỏng khi hai từ giống nhau

Khi `word1 == word2` thì nhánh `if w == word1` luôn bắt trước, nhánh `elif w == word2` **không bao giờ chạy**; `last2` mãi là `-1` và không có cặp nào được xét. Nếu dùng hai `if` liền nhau (không `elif`), cùng một chỉ số cập nhật cả `last1` lẫn `last2` và ta đo khoảng cách `0` giữa chỉ số với chính nó. Cần một cách xử lý riêng, hoặc cách thống nhất ở Cách 4.

In [6]:
def buggy_elif(words, word1, word2):
    last1 = last2 = -1
    best = len(words)
    for k, w in enumerate(words):
        if w == word1:
            last1 = k
        elif w == word2:
            last2 = k
        if last1 != -1 and last2 != -1:
            best = min(best, abs(last1 - last2))
    return best


def buggy_two_ifs(words, word1, word2):
    last1 = last2 = -1
    best = len(words)
    for k, w in enumerate(words):
        if w == word1:
            last1 = k
        if w == word2:
            last2 = k
        if last1 != -1 and last2 != -1:
            best = min(best, abs(last1 - last2))
    return best


print("elif   :", buggy_elif(WORDS, "makes", "makes"), "(không có cặp nào, còn nguyên giá trị khởi đầu n =", len(WORDS), ")")
print("hai if :", buggy_two_ifs(WORDS, "makes", "makes"), "(khoảng cách chỉ số với chính nó = 0)")

elif   : 5 (không có cặp nào, còn nguyên giá trị khởi đầu n = 5 )
hai if : 0 (khoảng cách chỉ số với chính nó = 0)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`wordsDict = ["practice", "makes", "perfect", "coding", "makes"]` (chỉ số `0..4`).

**Truy vấn `("makes", "coding")` (khác nhau):** `makes` ở chỉ số `1`, `4`; `coding` ở `3`. Khoảng cách `|1−3| = 2`, `|4−3| = 1`: đáp án `1`.

**Truy vấn `("makes", "makes")` (giống nhau):** `makes` ở chỉ số `1` và `4`. Chỉ có một cặp **khác nhau** `(1, 4)`: khoảng cách `3`. Cặp `(1, 1)` hay `(4, 4)` (cùng một lần xuất hiện) **không** tính: nếu tính thì đáp án là `0`, sai.

Điểm chung: đáp án là `min |i − j|` trên các cặp `(i, j)` với `words[i] == word1`, `words[j] == word2` và `i ≠ j`. Khi hai từ khác nhau điều kiện `i ≠ j` tự thỏa; khi giống nhau thì phải **chủ động loại** cặp `i == j`.

In [7]:
words = WORDS
for w1, w2 in [("makes", "coding"), ("makes", "makes")]:
    pairs = [(i, j, abs(i - j)) for i in range(len(words)) for j in range(len(words))
             if words[i] == w1 and words[j] == w2 and i != j]
    print(f"({w1}, {w2}):", pairs, "-> nhỏ nhất", min(p[2] for p in pairs))

(makes, coding): [(1, 3, 2), (4, 3, 1)] -> nhỏ nhất 1
(makes, makes): [(1, 4, 3), (4, 1, 3)] -> nhỏ nhất 3


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp chỉ số

**Ý tưởng.** Hai vòng `for` lồng nhau trên mọi cặp `(i, j)`; nếu `i ≠ j`, `words[i] == word1` và `words[j] == word2` thì cập nhật đáp án bằng `abs(i − j)`.

In [8]:
def shortest_bruteforce(words: list[str], word1: str, word2: str) -> int:
    n = len(words)
    best = n
    for i in range(n):
        if words[i] != word1:
            continue
        for j in range(n):
            if i != j and words[j] == word2:        # hai lần xuất hiện KHÁC NHAU
                best = min(best, abs(i - j))
    return best


for words, w1, w2, expected in examples:
    print(f"({w1}, {w2}) -> {shortest_bruteforce(words, w1, w2)} (kỳ vọng {expected})")

(makes, coding) -> 1 (kỳ vọng 1)
(makes, makes) -> 3 (kỳ vọng 3)


**Độ phức tạp (Cách 1).** Gọi `n = len(wordsDict)`. **Time `O(n²)`** (tệ nhất; nếu chỉ lặp trong các chỉ số khớp thì `O(a·b)` với `a`, `b` là số lần xuất hiện), **Space `O(1)`**. Với `n = 10⁵` là `10¹⁰` phép so sánh: **quá chậm**.

**Nút thắt.** Ta xét mọi cặp dù chỉ cần xét các cặp **gần nhau**. Chỉ số gần nhất đã gặp của mỗi từ là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

**Chỉ cần nhìn "lần xuất hiện gần nhất trước đó".** Cặp tối ưu `(i, j)` có `i < j` hoặc `i > j`. Giả sử `i < j` (đối xứng với trường hợp còn lại). Khi duyệt tới `j`, nếu `i` không phải lần xuất hiện của `word1` **gần nhất trước `j`** thì có `i' ∈ (i, j)` cũng là `word1` và `|j − i'| < |j − i|`, mâu thuẫn tính tối ưu. Vậy cặp tối ưu luôn gồm một phần tử và **lần xuất hiện gần nhất, trước đó, của từ còn lại**.

Do đó duyệt một lượt, giữ hai biến `last1`, `last2` (chỉ số gần nhất của `word1`, `word2`). Tại chỉ số `k`:

- nếu `words[k] == word1` thì cập nhật đáp án bằng `k − last2` (nếu `last2` tồn tại);
- nếu `words[k] == word2` thì cập nhật đáp án bằng `k − last1` (nếu `last1` tồn tại);
- **sau đó** mới cập nhật `last1 = k` (nếu khớp `word1`) và `last2 = k` (nếu khớp `word2`).

Thứ tự "kiểm tra bằng chỉ số *trước đó*, rồi mới ghi nhận chỉ số hiện tại" làm cho **cùng một đoạn code đúng cả khi `word1 == word2`**: lúc đó `last1 == last2 ==` lần xuất hiện trước, và hai điều kiện cùng cho `k − (lần trước)`; không bao giờ ra `0` vì ta chưa ghi nhận `k`.

In [9]:
def trace_unified(words, word1, word2):
    last1 = last2 = -1
    best = len(words)
    for k, w in enumerate(words):
        note = []
        if w == word1 and last2 != -1:
            best = min(best, k - last2)
            note.append(f"k - last2 = {k - last2}")
        if w == word2 and last1 != -1:
            best = min(best, k - last1)
            note.append(f"k - last1 = {k - last1}")
        if w == word1:
            last1 = k
        if w == word2:
            last2 = k
        print(f"  k={k} {w!r:11}: {', '.join(note) or '-':<24} last1={last1} last2={last2} best={best}")
    return best


print("makes, makes:")
assert trace_unified(WORDS, "makes", "makes") == 3

makes, makes:
  k=0 'practice' : -                        last1=-1 last2=-1 best=5
  k=1 'makes'    : -                        last1=1 last2=1 best=5
  k=2 'perfect'  : -                        last1=1 last2=1 best=5
  k=3 'coding'   : -                        last1=1 last2=1 best=5
  k=4 'makes'    : k - last2 = 3, k - last1 = 3 last1=4 last2=4 best=3


## 7. Các cách tiếp theo

### Cách 2 — Danh sách chỉ số, tách hai trường hợp

**Giải thích.** Tính `a = chỉ số của word1`. Nếu `word1 == word2`: đáp án là `min` các hiệu của hai phần tử **liên tiếp** của `a` (`a[t+1] − a[t]`). Ngược lại tính `b = chỉ số của word2` rồi tìm khoảng cách nhỏ nhất giữa hai danh sách đã sắp bằng **hai con trỏ** (tăng con trỏ của chỉ số nhỏ hơn).

In [10]:
def shortest_positions(words: list[str], word1: str, word2: str) -> int:
    a = [i for i, w in enumerate(words) if w == word1]
    if word1 == word2:
        return min(a[t + 1] - a[t] for t in range(len(a) - 1))      # hai lần xuất hiện liên tiếp
    b = [j for j, w in enumerate(words) if w == word2]
    i = j = 0
    best = len(words)
    while i < len(a) and j < len(b):
        best = min(best, abs(a[i] - b[j]))
        if a[i] < b[j]:
            i += 1
        else:
            j += 1
    return best


for words, w1, w2, expected in examples:
    print(f"({w1}, {w2}) -> {shortest_positions(words, w1, w2)} (kỳ vọng {expected})")

(makes, coding) -> 1 (kỳ vọng 1)
(makes, makes) -> 3 (kỳ vọng 3)


**Độ phức tạp Cách 2:** **Time `O(n)`** (tạo hai danh sách rồi duyệt), **Space `O(a + b)`** ≤ `O(n)` cho hai danh sách.

### Cách 3 — Một lượt duyệt, tách hai trường hợp (cách của lời giải tham khảo)

**Giải thích.** Nếu `word1 == word2`: giữ **một** chỉ số `j` (lần xuất hiện trước); mỗi lần gặp từ đó tại `i`, cập nhật `best = min(best, i − j)` (nếu `j ≠ −1`), rồi `j = i`. Nếu `word1 ≠ word2`: giữ hai chỉ số `i`, `j`; mỗi khi gặp `word1` cập nhật `i`, gặp `word2` cập nhật `j`, rồi nếu cả hai `≠ −1` thì `best = min(best, |i − j|)`. `O(1)` bộ nhớ phụ.

In [11]:
def shortest_two_cases(words: list[str], word1: str, word2: str) -> int:
    ans = len(words)
    if word1 == word2:
        j = -1                                          # lần xuất hiện trước của word1
        for i, w in enumerate(words):
            if w == word1:
                if j != -1:
                    ans = min(ans, i - j)
                j = i
    else:
        i = j = -1                                      # chỉ số gần nhất của word1, word2
        for k, w in enumerate(words):
            if w == word1:
                i = k
            if w == word2:
                j = k
            if i != -1 and j != -1:
                ans = min(ans, abs(i - j))
    return ans


for words, w1, w2, expected in examples:
    print(f"({w1}, {w2}) -> {shortest_two_cases(words, w1, w2)} (kỳ vọng {expected})")

(makes, coding) -> 1 (kỳ vọng 1)
(makes, makes) -> 3 (kỳ vọng 3)


In [12]:
# Code nộp LeetCode cho Cách 3
from typing import List


class Solution:
    def shortestWordDistance(self, wordsDict: List[str], word1: str, word2: str) -> int:
        ans = len(wordsDict)
        if word1 == word2:
            j = -1
            for i, w in enumerate(wordsDict):
                if w == word1:
                    if j != -1:
                        ans = min(ans, i - j)
                    j = i
        else:
            i = j = -1
            for k, w in enumerate(wordsDict):
                if w == word1:
                    i = k
                if w == word2:
                    j = k
                if i != -1 and j != -1:
                    ans = min(ans, abs(i - j))
        return ans

**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Một lượt duyệt thống nhất: "kiểm tra bằng chỉ số trước, cập nhật sau"

**Giải thích.** Không cần rẽ nhánh `word1 == word2`: đoạn code ở mục 6 xử lý cả hai trường hợp. Tại mỗi `k`: (1) nếu gặp `word1` và đã từng thấy `word2` thì cập nhật bằng `k − last2`; (2) nếu gặp `word2` và đã từng thấy `word1` thì cập nhật bằng `k − last1`; (3) **cuối cùng** cập nhật `last1`, `last2`. Bước (3) đứng sau cùng là điểm mấu chốt.

In [13]:
def shortest_unified(words: list[str], word1: str, word2: str) -> int:
    ans = len(words)
    last1 = last2 = -1                                 # chỉ số gần nhất TRƯỚC k của word1, word2
    for k, w in enumerate(words):
        if w == word1 and last2 != -1:
            ans = min(ans, k - last2)                  # word1 ở k, word2 gần nhất ở last2 < k
        if w == word2 and last1 != -1:
            ans = min(ans, k - last1)                  # word2 ở k, word1 gần nhất ở last1 < k
        if w == word1:
            last1 = k                                  # cập nhật SAU khi đã so sánh
        if w == word2:
            last2 = k
    return ans


for words, w1, w2, expected in examples:
    print(f"({w1}, {w2}) -> {shortest_unified(words, w1, w2)} (kỳ vọng {expected})")

(makes, coding) -> 1 (kỳ vọng 1)
(makes, makes) -> 3 (kỳ vọng 3)


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

> Cách 3 là cách nộp bài quen thuộc. Cách 4 là cách viết gọn, không rẽ nhánh. Cách 1 và 2 làm chuẩn đối chiếu và để thấy cấu trúc "chỉ số xuất hiện".

## 8. TRACE từng bước

Ta theo dõi Cách 4 (thống nhất) với hai truy vấn của đề, và một mảng dài hơn để thấy chỉ số gần nhất thay đổi.

In [14]:
def trace_all(words, word1, word2):
    print(f"words={words}, word1={word1!r}, word2={word2!r}")
    result = trace_unified(words, word1, word2)
    print("  => đáp án:", result)
    return result


assert trace_all(WORDS, "makes", "coding") == 1
print()
assert trace_all(["a", "b", "x", "a", "a", "b"], "a", "a") == 1

words=['practice', 'makes', 'perfect', 'coding', 'makes'], word1='makes', word2='coding'
  k=0 'practice' : -                        last1=-1 last2=-1 best=5
  k=1 'makes'    : -                        last1=1 last2=-1 best=5
  k=2 'perfect'  : -                        last1=1 last2=-1 best=5
  k=3 'coding'   : k - last1 = 2            last1=1 last2=3 best=2
  k=4 'makes'    : k - last2 = 1            last1=4 last2=3 best=1
  => đáp án: 1

words=['a', 'b', 'x', 'a', 'a', 'b'], word1='a', word2='a'
  k=0 'a'        : -                        last1=0 last2=0 best=6
  k=1 'b'        : -                        last1=0 last2=0 best=6
  k=2 'x'        : -                        last1=0 last2=0 best=6
  k=3 'a'        : k - last2 = 3, k - last1 = 3 last1=3 last2=3 best=3
  k=4 'a'        : k - last2 = 1, k - last1 = 1 last1=4 last2=4 best=1
  k=5 'b'        : -                        last1=4 last2=4 best=1
  => đáp án: 1


Bảng tóm tắt (Cách 4, `("makes", "makes")`, `words = [practice, makes, perfect, coding, makes]`):

| `k` | `words[k]` | cập nhật đáp án | `last1` | `last2` | `best` |
|---|---|---|---|---|---|
| 0 | `practice` | không | -1 | -1 | 5 |
| 1 | `makes` | không (`last` còn `-1`) | 1 | 1 | 5 |
| 2 | `perfect` | không | 1 | 1 | 5 |
| 3 | `coding` | không | 1 | 1 | 5 |
| 4 | `makes` | `k − last2 = 3` và `k − last1 = 3` | 4 | 4 | 3 |

Đáp án `3`, đúng với đề.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (hai trường hợp), đúng class `Solution`, đúng method `shortestWordDistance(self, wordsDict, word1, word2)`.

In [15]:
from typing import List


class Solution:
    def shortestWordDistance(self, wordsDict: List[str], word1: str, word2: str) -> int:
        ans = len(wordsDict)
        if word1 == word2:                       # cùng một từ: khoảng cách giữa hai lần xuất hiện liên tiếp
            j = -1
            for i, w in enumerate(wordsDict):
                if w == word1:
                    if j != -1:
                        ans = min(ans, i - j)
                    j = i
        else:                                    # hai từ khác nhau: chỉ số gần nhất của mỗi từ
            i = j = -1
            for k, w in enumerate(wordsDict):
                if w == word1:
                    i = k
                if w == word2:
                    j = k
                if i != -1 and j != -1:
                    ans = min(ans, abs(i - j))
        return ans


for words, w1, w2, expected in examples:
    assert Solution().shortestWordDistance(words, w1, w2) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Cặp tối ưu dùng lần xuất hiện gần nhất:** gọi cặp tối ưu `(i, j)`, `i ≠ j`. Giả sử `i < j` (trường hợp `i > j` đối xứng). Nếu có `i'` với `i < i' < j` và `words[i'] == word1` thì `(i', j)` cũng hợp lệ (`i' ≠ j`) và gần hơn, mâu thuẫn. Vậy `i` là lần xuất hiện của `word1` **gần nhất trước `j`**; tại bước `k = j` biến `last1` đúng bằng `i`.
- **Hai từ khác nhau (Cách 3, nhánh else):** `i` luôn cập nhật khi gặp `word1`, `j` khi gặp `word2`; mỗi lần cả hai đã xác định, `|i − j|` là khoảng cách giữa hai lần xuất hiện gần nhất; mọi cặp tối ưu (theo chứng minh trên) được xét ở thời điểm xuất hiện của phần tử đứng sau.
- **Hai từ giống nhau (Cách 3, nhánh if):** cặp `(i, j)` với `i ≠ j` đều là hai lần xuất hiện khác nhau của cùng một từ; khoảng cách nhỏ nhất đạt ở hai lần xuất hiện **liên tiếp** (nếu có lần ở giữa, chia đôi khoảng cách). Duyệt giữ `j` = lần xuất hiện trước và tính `i − j` bao phủ mọi cặp liên tiếp.
- **Cách 4:** tại `k` ta so sánh với `last1`, `last2` **trước khi** cập nhật chúng, nên các chỉ số này đều `< k`; không bao giờ so `k` với chính `k`. Khi `word1 == word2`, `last1 == last2 =` lần xuất hiện trước đó, hai phép so sánh cho cùng giá trị `k − trước`.
- **Đảm bảo của đề:** `word1`, `word2` có trong mảng, và khi trùng nhau thì từ xuất hiện ít nhất hai lần, nên có ít nhất một cặp hợp lệ: đáp án luôn được cập nhật từ giá trị khởi đầu `n`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(wordsDict)`, `a`, `b` là số lần xuất hiện của `word1`, `word2`.

- **Cách 1 (mọi cặp chỉ số):** Time `O(n²)` (hoặc `O(a·b)` nếu bỏ qua chỉ số không khớp), Space `O(1)`.
- **Cách 2 (danh sách chỉ số):** Time `O(n)`, Space `O(a + b)`.
- **Cách 3 (một lượt, hai trường hợp):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (một lượt, thống nhất):** Time `O(n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất như nhau (luôn duyệt hết mảng). Không thể nhanh hơn `O(n)` vì phải đọc hết dữ liệu (không có tiền xử lý như bài II).

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` viết riêng theo định nghĩa: với mọi cặp `i < j`, nếu `{words[i], words[j]}` khớp `{word1, word2}` (đúng cả khi hai từ trùng nhau: cả hai phần tử phải bằng từ đó) thì tính `j − i`. Dữ liệu ngẫu nhiên đảm bảo `word1`, `word2` có mặt và, khi trùng nhau, xuất hiện ít nhất hai lần.

In [16]:
sol = Solution()
impls = {
    "bruteforce": shortest_bruteforce,
    "positions": shortest_positions,
    "two_cases": shortest_two_cases,
    "unified": shortest_unified,
    "submit": sol.shortestWordDistance,
}


def oracle(words, word1, word2):
    best = None
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            pair = (words[i], words[j])
            if pair == (word1, word2) or pair == (word2, word1):
                d = j - i
                best = d if best is None else min(best, d)
    return best


def valid_query(words, w1, w2):
    return (w1 in words and w2 in words and (w1 != w2 or words.count(w1) >= 2))


for words, w1, w2, expected in examples:
    assert oracle(words, w1, w2) == expected
    for name, fn in impls.items():
        assert fn(list(words), w1, w2) == expected, (name, w1, w2, fn(words, w1, w2), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [17]:
def check_all(words, w1, w2, label, skip=(), expected=None):
    assert valid_query(words, w1, w2), (label, "truy vấn không hợp lệ")
    want = oracle(words, w1, w2) if len(words) <= 400 else None
    if want is None:
        want = shortest_unified(words, w1, w2)           # dữ liệu lớn: dùng bản thống nhất, đã được kiểm chứng ở trên
    if expected is not None:
        assert want == expected, (label, "oracle", want, expected)
    for name, fn in impls.items():
        if name in skip:
            continue
        original = list(words)
        got = fn(original, w1, w2)
        assert got == want, (label, name, got, want)
        assert original == words, (label, name, "đã sửa đầu vào")
    print("PASS", label, "->", want)


check_all(["a", "b"], "a", "b", "n = 2, kề nhau", expected=1)
check_all(["a", "b"], "b", "a", "n = 2, đảo thứ tự", expected=1)
check_all(["a", "a"], "a", "a", "n = 2, cùng một từ", expected=1)
check_all(["a", "x", "a"], "a", "a", "cùng một từ, cách 2", expected=2)
check_all(["a", "x", "x", "a", "a"], "a", "a", "cùng một từ: lấy khoảng cách NHỎ NHẤT giữa các lần liên tiếp (1)", expected=1)
check_all(["a", "x", "b"], "a", "b", "cách 2", expected=2)
check_all(["b", "x", "x", "a", "b"], "a", "b", "từ b ở hai phía của a", expected=1)
check_all(["x", "a", "x", "x", "b"], "a", "b", "lỗi 'khởi tạo chỉ số bằng 0' bị bắt", expected=3)
check_all(["m", "a", "x", "x", "a", "a"], "a", "a", "lỗi 'chỉ xét hai lần đầu' bị bắt", expected=1)
check_all(WORDS, "makes", "makes", "ví dụ 2 của đề", expected=3)
check_all(WORDS, "coding", "makes", "khác nhau, đảo thứ tự", expected=1)
check_all(["a"] * 20, "a", "a", "toàn một từ", expected=1)
big = ["a"] + ["x"] * 99998 + ["a"]
check_all(big, "a", "a", "n = 10^5: hai lần xuất hiện ở hai đầu", skip=("bruteforce",), expected=99999)
check_all(["a", "b"] * 50000, "a", "b", "n = 10^5: xen kẽ", skip=("bruteforce",), expected=1)

PASS n = 2, kề nhau -> 1
PASS n = 2, đảo thứ tự -> 1
PASS n = 2, cùng một từ -> 1
PASS cùng một từ, cách 2 -> 2
PASS cùng một từ: lấy khoảng cách NHỎ NHẤT giữa các lần liên tiếp (1) -> 1
PASS cách 2 -> 2
PASS từ b ở hai phía của a -> 1
PASS lỗi 'khởi tạo chỉ số bằng 0' bị bắt -> 3
PASS lỗi 'chỉ xét hai lần đầu' bị bắt -> 1
PASS ví dụ 2 của đề -> 3
PASS khác nhau, đảo thứ tự -> 1
PASS toàn một từ -> 1
PASS n = 10^5: hai lần xuất hiện ở hai đầu -> 99999
PASS n = 10^5: xen kẽ -> 1


### 12.2. Tính chất và phản ví dụ

In [18]:
random.seed(245)


def random_case():
    n = random.randint(2, 30)
    alphabet = random.choice(["ab", "abc", "abcd"])
    words = [random.choice(alphabet) for _ in range(n)]
    candidates = [(a, b) for a in set(words) for b in set(words) if valid_query(words, a, b)]
    if not candidates:
        return None
    return words, random.choice(candidates)


# Tính chất 1: đối xứng shortest(w1, w2) == shortest(w2, w1); 1 <= đáp án <= n - 1
for _ in range(500):
    case = random_case()
    if case is None:
        continue
    words, (w1, w2) = case
    a, b = sol.shortestWordDistance(words, w1, w2), sol.shortestWordDistance(words, w2, w1)
    assert a == b and 1 <= a <= len(words) - 1
print("PASS tính chất: đối xứng và 1 <= đáp án <= n - 1")

# Tính chất 2: với w1 == w2, đáp án = khoảng cách nhỏ nhất giữa các chỉ số liên tiếp (độc lập với cách làm)
for _ in range(300):
    n = random.randint(2, 30)
    words = [random.choice("ab") for _ in range(n)]
    for w in "ab":
        pos = [i for i, x in enumerate(words) if x == w]
        if len(pos) >= 2:
            assert sol.shortestWordDistance(words, w, w) == min(y - x for x, y in zip(pos, pos[1:]))
print("PASS tính chất: cùng một từ -> hiệu nhỏ nhất của các chỉ số liên tiếp")

# Phản ví dụ: dùng code của bài 'hai từ khác nhau' cho trường hợp bằng nhau
print("(makes, makes) -> sai (elif):", buggy_elif(WORDS, "makes", "makes"), "| sai (hai if):", buggy_two_ifs(WORDS, "makes", "makes"),
      "| đúng:", sol.shortestWordDistance(WORDS, "makes", "makes"))
assert buggy_elif(WORDS, "makes", "makes") == 5 and buggy_two_ifs(WORDS, "makes", "makes") == 0

PASS tính chất: đối xứng và 1 <= đáp án <= n - 1
PASS tính chất: cùng một từ -> hiệu nhỏ nhất của các chỉ số liên tiếp
(makes, makes) -> sai (elif): 5 | sai (hai if): 0 | đúng: 3


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [19]:
random.seed(2026)
done = 0
for trial in range(3000):
    case = random_case()
    if case is None:
        continue
    words, (w1, w2) = case
    expected = oracle(words, w1, w2)
    for name, fn in impls.items():
        assert fn(list(words), w1, w2) == expected, (name, words, w1, w2)
    done += 1

for trial in range(10):                           # mảng dài: so các cách O(n) và danh sách chỉ số
    n = random.randint(20000, 100000)
    vocab = [f"w{i}" for i in range(random.choice([2, 5, 50]))]
    words = [random.choice(vocab) for _ in range(n)]
    for w1, w2 in [(vocab[0], vocab[1]), (vocab[0], vocab[0]), (vocab[1], vocab[1])]:
        if valid_query(words, w1, w2):
            results = {name: fn(list(words), w1, w2) for name, fn in impls.items() if name != "bruteforce"}
            assert len(set(results.values())) == 1, (n, w1, w2, results)
print("PASS", done, "truy vấn ngẫu nhiên khớp oracle độc lập; 10 mảng dài khớp giữa các cách O(n)")

PASS 3000 truy vấn ngẫu nhiên khớp oracle độc lập; 10 mảng dài khớp giữa các cách O(n)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng code của bài "hai từ khác nhau" khi `word1 == word2`

In [20]:
print("(makes, makes) -> hai if   :", buggy_two_ifs(WORDS, "makes", "makes"), "| đúng:", Solution().shortestWordDistance(WORDS, "makes", "makes"))
print("(makes, makes) -> elif     :", buggy_elif(WORDS, "makes", "makes"), "| đúng:", Solution().shortestWordDistance(WORDS, "makes", "makes"))

(makes, makes) -> hai if   : 0 | đúng: 3
(makes, makes) -> elif     : 5 | đúng: 3


**Vì sao sai:** khi hai từ giống nhau, `if w == word1: i = k` và `if w == word2: j = k` cùng chạy ở mọi vị trí của từ đó, nên `i == j` và ta đo khoảng cách `0` (hai `if`); còn với `elif` thì `j` không bao giờ được cập nhật và đáp án giữ nguyên giá trị khởi đầu `n`. Phải xử lý trường hợp trùng nhau riêng (một biến `last`) hoặc dùng cách "so sánh trước, cập nhật sau" (Cách 4).

### Lỗi 2: Khởi tạo chỉ số bằng `0` thay vì `-1`

In [21]:
def sai_khoi_tao_0(words, word1, word2):
    ans = len(words)
    i = j = 0                                  # sai: 0 là chỉ số hợp lệ, không phải "chưa thấy"
    for k, w in enumerate(words):
        if w == word1:
            i = k
            ans = min(ans, abs(i - j))
        elif w == word2:
            j = k
            ans = min(ans, abs(i - j))
    return ans


words = ["x", "a", "x", "x", "b"]
print("sai:", sai_khoi_tao_0(words, "a", "b"), "| đúng:", Solution().shortestWordDistance(words, "a", "b"))

sai: 1 | đúng: 3


**Vì sao sai:** `0` là một chỉ số hợp lệ nên "chưa gặp từ kia" bị nhầm với "từ kia ở chỉ số 0". Ở ví dụ, khi gặp `a` ở chỉ số `1` ta so với `j = 0` (giá trị khởi tạo giả) và ra khoảng cách `1`, trong khi `b` thật ở chỉ số `4`. Dùng giá trị **không thể là chỉ số** (`-1`) hoặc `None` làm dấu "chưa thấy", và kiểm tra trước khi dùng.

### Lỗi 3: Chỉ xét hai lần xuất hiện đầu tiên khi hai từ giống nhau

In [22]:
def sai_hai_lan_dau(words, word1, word2):
    pos = [i for i, w in enumerate(words) if w == word1]
    return pos[1] - pos[0]                      # sai: chỉ lấy hai lần đầu tiên


words = ["m", "a", "x", "x", "a", "a"]
print("sai:", sai_hai_lan_dau(words, "a", "a"), "| đúng:", Solution().shortestWordDistance(words, "a", "a"))

sai: 3 | đúng: 1


**Vì sao sai:** khoảng cách nhỏ nhất có thể ở **bất kỳ** cặp lần xuất hiện liên tiếp nào, không nhất thiết hai lần đầu. Ở ví dụ `a` ở chỉ số `1, 4, 5`: hai lần đầu cách `3`, nhưng `4` và `5` cách `1`. Phải xét mọi hiệu liên tiếp.

### Lỗi 4: Cập nhật chỉ số trước khi so sánh (trong cách thống nhất)

In [23]:
def sai_cap_nhat_truoc(words, word1, word2):
    ans = len(words)
    last1 = last2 = -1
    for k, w in enumerate(words):
        if w == word1:
            last1 = k                           # cập nhật TRƯỚC
        if w == word2:
            last2 = k
        if w == word1 and last2 != -1:
            ans = min(ans, k - last2)           # lúc này last2 có thể đã là k (khi word1 == word2) -> 0
        if w == word2 and last1 != -1:
            ans = min(ans, k - last1)
    return ans


print("(makes, makes) -> sai:", sai_cap_nhat_truoc(WORDS, "makes", "makes"), "| đúng:", Solution().shortestWordDistance(WORDS, "makes", "makes"))

(makes, makes) -> sai: 0 | đúng: 3


**Vì sao sai:** khi cập nhật `last1`, `last2` bằng `k` **trước** lúc so sánh, hiệu `k − last` thành `0` nếu hai từ trùng nhau. Trật tự đúng: **so sánh với chỉ số trước đó, rồi mới ghi nhận chỉ số hiện tại**.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Khoảng cách ngắn nhất giữa hai lần xuất hiện" → duyệt một lượt, ghi nhớ **chỉ số xuất hiện gần nhất** của mỗi từ: `O(n)` thời gian, `O(1)` bộ nhớ.
- Khi hai "vai" có thể trùng nhau, dùng nguyên tắc **"kiểm tra trước, cập nhật sau"** để cùng một đoạn code xử lý cả hai trường hợp.
- Dùng `-1` (hoặc `None`) làm dấu "chưa gặp", không dùng `0`.
- Nhiều truy vấn trên cùng một mảng → tiền xử lý danh sách chỉ số (bài II: `WordDistance`).
- Trường hợp `word1 == word2`: khoảng cách nhỏ nhất luôn giữa hai lần xuất hiện **liên tiếp**.
- Test: oracle duyệt mọi cặp `i < j` và kiểm tra tập `{words[i], words[j]}` trùng `{word1, word2}`.

## 15. Biến thể của bài

1. **Shortest Word Distance I (LeetCode 243):** `word1 ≠ word2`, một truy vấn.
2. **Shortest Word Distance II (LeetCode 244):** nhiều truy vấn, thiết kế lớp (bài kế trước).
3. **Khoảng cách xa nhất** giữa hai lần xuất hiện của hai từ (cẩn thận trường hợp trùng nhau).
4. **Trả về cặp chỉ số** `(i, j)` đạt khoảng cách nhỏ nhất.
5. **Đếm số cặp** `(i, j)` có khoảng cách `≤ d`.

In [24]:
def farthest_distance(words, word1, word2):
    """Biến thể 3: khoảng cách lớn nhất giữa hai lần xuất hiện KHÁC NHAU."""
    pos1 = [i for i, w in enumerate(words) if w == word1]
    pos2 = [j for j, w in enumerate(words) if w == word2]
    if word1 == word2:
        return pos1[-1] - pos1[0]
    return max(abs(pos1[0] - pos2[-1]), abs(pos1[-1] - pos2[0]))


def closest_pair(words, word1, word2):
    """Biến thể 4: (i, j) với i < j đạt khoảng cách nhỏ nhất."""
    best, pair = len(words), None
    last1 = last2 = -1
    for k, w in enumerate(words):
        if w == word1 and last2 != -1 and k - last2 < best:
            best, pair = k - last2, (last2, k)
        if w == word2 and last1 != -1 and k - last1 < best:
            best, pair = k - last1, (last1, k)
        if w == word1:
            last1 = k
        if w == word2:
            last2 = k
    return pair


def count_pairs_within(words, word1, word2, d):
    """Biến thể 5: số cặp (i, j), i < j, chỉ số khác nhau, tập {words[i], words[j]} trùng {word1, word2}, j - i <= d."""
    count = 0
    n = len(words)
    for i in range(n):
        for j in range(i + 1, min(n, i + d + 1)):
            if (words[i], words[j]) in {(word1, word2), (word2, word1)}:
                count += 1
    return count


assert farthest_distance(WORDS, "makes", "makes") == 3
assert farthest_distance(WORDS, "makes", "practice") == 4
assert closest_pair(WORDS, "makes", "coding") == (3, 4)
assert closest_pair(WORDS, "makes", "makes") == (1, 4)
assert count_pairs_within(WORDS, "makes", "coding", 1) == 1 and count_pairs_within(WORDS, "makes", "makes", 3) == 1
random.seed(15)
for _ in range(200):
    case = random_case()
    if case is None:
        continue
    words, (w1, w2) = case
    i, j = closest_pair(words, w1, w2)
    assert j - i == oracle(words, w1, w2)
    assert count_pairs_within(words, w1, w2, oracle(words, w1, w2)) >= 1
print("PASS biến thể: xa nhất, cặp chỉ số, đếm cặp trong khoảng d")

PASS biến thể: xa nhất, cặp chỉ số, đếm cặp trong khoảng d


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp chỉ số | hai vòng `for`, loại `i == j` | `O(n²)` | `O(1)` | hiểu đề; oracle |
| 2. Danh sách chỉ số | hiệu liên tiếp (trùng) / hai con trỏ (khác) | `O(n)` | `O(a + b)` | nền tảng cho bài II |
| 3. Một lượt, hai trường hợp | `last` đơn hoặc `i`, `j` | `O(n)` | `O(1)` | **cách nộp bài** |
| 4. Một lượt thống nhất | so sánh trước, cập nhật sau | `O(n)` | `O(1)` | gọn, không rẽ nhánh `word1 == word2` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `abs(2 - 6)`, `list(enumerate(["a", "b"]))`, `Solution().shortestWordDistance(["a", "b", "a"], "a", "b")`, `Solution().shortestWordDistance(["a", "b", "a"], "a", "a")`, `Solution().shortestWordDistance(["a", "x", "b"], "b", "a")`.

**Bài 2 (tính tay).** Với `words = ["a", "x", "b", "a", "b", "a", "x"]`: tìm khoảng cách ngắn nhất cho `("a", "b")` và `("a", "a")` và `("b", "b")`. Liệt kê các chỉ số của từng từ.

**Bài 3 (điền TODO).** Hoàn thành `shortest_todo(words, word1, word2)` theo Cách 4 (thống nhất).

In [25]:
def shortest_todo(words, word1, word2):
    ans = len(words)
    last1 = last2 = -1
    for k, w in enumerate(words):
        # TODO 1: nếu w == word1 và last2 != -1: ans = min(ans, k - last2)
        # TODO 2: nếu w == word2 và last1 != -1: ans = min(ans, k - last1)
        # TODO 3: SAU CÙNG: nếu w == word1 thì last1 = k; nếu w == word2 thì last2 = k
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `shortest_positions2(words, word1, word2)` theo Cách 2 (danh sách chỉ số; hiệu liên tiếp khi trùng nhau, hai con trỏ khi khác nhau). Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `shortest_any(words, targets)` trả về khoảng cách ngắn nhất giữa hai vị trí **khác nhau** bất kỳ có từ thuộc tập `targets` (`len(targets) ≥ 1`; nếu `targets` chỉ có một từ thì cần hai lần xuất hiện của từ đó). (b) Viết `shortest_k_apart(words, word, k)` trả `True` nếu mọi hai lần xuất hiện liên tiếp của `word` cách nhau ít nhất `k`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `abs` là giá trị tuyệt đối; khi hai từ trùng nhau cần hai lần xuất hiện khác nhau. Bài 2: viết các chỉ số xuất hiện của `a`, `b`.
- **Mức 2 (vừa):** Bài 1: `4`, `[(0, 'a'), (1, 'b')]`, `1`, `2`, `2`. Bài 2: `a` ở `[0, 3, 5]`, `b` ở `[2, 4]`: `("a","b")`: min của `|0-2|, |0-4|, |3-2|, |3-4|, |5-2|, |5-4|` là `1`; `("a","a")`: các hiệu liên tiếp `3, 2` nên `2`; `("b","b")`: `4 − 2 = 2`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: nhánh trùng nhau dùng `min(a[t+1] - a[t])`; nhánh khác nhau: hai con trỏ như bài II. Bài 5(a): giữ `last` là chỉ số gần nhất của **bất kỳ** từ thuộc `targets`; mỗi lần gặp một từ thuộc `targets` cập nhật `ans = min(ans, k − last)` (nếu `last ≠ −1`) rồi `last = k`. Bài 5(b): duyệt lấy `last` như nhánh "trùng nhau" và so với `k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert abs(2 - 6) == 4
assert list(enumerate(["a", "b"])) == [(0, "a"), (1, "b")]
assert Solution().shortestWordDistance(["a", "b", "a"], "a", "b") == 1
assert Solution().shortestWordDistance(["a", "b", "a"], "a", "a") == 2
assert Solution().shortestWordDistance(["a", "x", "b"], "b", "a") == 2

# Bài 2
w2 = ["a", "x", "b", "a", "b", "a", "x"]
assert [i for i, w in enumerate(w2) if w == "a"] == [0, 3, 5]
assert [i for i, w in enumerate(w2) if w == "b"] == [2, 4]
assert Solution().shortestWordDistance(w2, "a", "b") == 1
assert Solution().shortestWordDistance(w2, "a", "a") == 2
assert Solution().shortestWordDistance(w2, "b", "b") == 2

# Bài 3
def shortest_todo_ans(words, word1, word2):
    ans = len(words)
    last1 = last2 = -1
    for k, w in enumerate(words):
        if w == word1 and last2 != -1:
            ans = min(ans, k - last2)
        if w == word2 and last1 != -1:
            ans = min(ans, k - last1)
        if w == word1:
            last1 = k
        if w == word2:
            last2 = k
    return ans

for words, w1, w2, expected in examples:
    assert shortest_todo_ans(words, w1, w2) == expected

# Bài 4
def shortest_positions2(words, word1, word2):
    a = [i for i, w in enumerate(words) if w == word1]
    if word1 == word2:
        return min(a[t + 1] - a[t] for t in range(len(a) - 1))
    b = [j for j, w in enumerate(words) if w == word2]
    i = j = 0
    best = len(words)
    while i < len(a) and j < len(b):
        best = min(best, abs(a[i] - b[j]))
        if a[i] < b[j]:
            i += 1
        else:
            j += 1
    return best

for words, w1, w2, expected in examples:
    assert shortest_positions2(words, w1, w2) == expected

# Bài 5(a), 5(b)
def shortest_any(words, targets):
    ans, last = len(words), -1
    for k, w in enumerate(words):
        if w in targets:
            if last != -1:
                ans = min(ans, k - last)
            last = k
    return ans

def shortest_k_apart(words, word, k):
    last = -1
    for i, w in enumerate(words):
        if w == word:
            if last != -1 and i - last < k:
                return False
            last = i
    return True

assert shortest_any(WORDS, {"makes", "coding"}) == 1
assert shortest_any(WORDS, {"makes"}) == 3
assert shortest_k_apart(WORDS, "makes", 3) is True and shortest_k_apart(WORDS, "makes", 4) is False
random.seed(19)
for _ in range(200):
    case = random_case()
    if case is None:
        continue
    words, (w1, w2) = case
    assert shortest_any(words, {w1, w2}) <= sol.shortestWordDistance(words, w1, w2)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một lượt, so sánh với chỉ số trước đó rồi mới cập nhật)

```python
class Solution:
    def shortestWordDistance(self, wordsDict: List[str], word1: str, word2: str) -> int:
        ans = len(wordsDict)
        last1 = last2 = -1
        for k, w in enumerate(wordsDict):
            if w == word1 and last2 != -1:
                ans = min(ans, k - last2)
            if w == word2 and last1 != -1:
                ans = min(ans, k - last1)
            if w == word1:
                last1 = k
            if w == word2:
                last2 = k
        return ans
```

**Checklist**

- [ ] Hiểu `word1` có thể **bằng** `word2`: cần hai lần xuất hiện **khác nhau** (không tính `i == j`)?
- [ ] Dùng `-1` (hoặc `None`) làm dấu "chưa gặp", không dùng `0`?
- [ ] **So sánh trước, cập nhật sau** (hoặc tách hai trường hợp rõ ràng)?
- [ ] Không dùng `elif` giữa hai điều kiện `w == word1` và `w == word2` khi chúng có thể cùng đúng?
- [ ] Khi trùng nhau lấy hiệu của **mọi** cặp liên tiếp (không chỉ hai lần đầu)?
- [ ] Đã thử: `n = 2`, cùng một từ ở hai đầu, từ lặp nhiều lần, hai chiều `(a, b)` và `(b, a)`?
- [ ] Đã đối chiếu với oracle duyệt mọi cặp `i < j` trên mảng ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def shortestWordDistance(self, wordsDict: List[str], word1: str, word2: str) -> int`?